## This is the code to train the model and acquire influence for Number of Samples Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [3]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [4]:
import random
from keras.optimizers import SGD

In [5]:
from sklearn.datasets import make_classification
from sklearn.datasets import make_blobs

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to test on different number of samples.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [6]:
total_size = 8500
test_size = 500
n_features=10
seed=42
sep= 1.5
std0 = 1.0          
std1_dense = 0.01   
std1_sparse = 3.0

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [7]:
n_per_class = total_size // 2

In [8]:
big_size = n_per_class // 2
small_size = n_per_class - big_size

In [9]:
c0 = np.zeros(n_features)
c0[0] = -sep

c1 = np.zeros(n_features)
c1[0] = sep

In [10]:
X0_dense, _ = make_blobs(
    n_samples=small_size,
    centers=[c0],
    cluster_std=std1_dense,
    n_features=n_features,
    random_state=seed
)

X0_sparse, _ = make_blobs(
    n_samples=big_size,
    centers=[c0],
    cluster_std=std1_sparse,
    n_features=n_features,
    random_state=seed + 1
)

X1_dense, _ = make_blobs(
    n_samples=small_size,
    centers=[c1],
    cluster_std=std1_dense,
    n_features=n_features,
    random_state=seed + 2
)

X1_sparse, _ = make_blobs(
    n_samples=big_size,
    centers=[c1],
    cluster_std=std1_sparse,
    n_features=n_features,
    random_state=seed + 3
)

In [11]:
X = np.vstack([X0_dense, X0_sparse, X1_dense, X1_sparse])
y = np.hstack([
    np.zeros(big_size, dtype=int),
    np.zeros(small_size, dtype=int),
    np.ones(big_size, dtype=int),
    np.ones(small_size, dtype=int),
])

In [12]:
cluster_id = (
    (["0_dense"] * big_size) +
    (["0_sparse"] * small_size) +
    (["1_dense"] * big_size) +
    (["1_sparse"] * small_size)
)
rng = np.random.RandomState(seed)
idx = rng.permutation(X.shape[0])
X = X[idx]; y = y[idx]
cluster_id = np.array(cluster_id, dtype=object)[idx]

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [13]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
df["cluster_id"] = cluster_id 
print(df)
print(df["label"].value_counts())

      feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      1.504080  -0.000461   0.008391  -0.004641  -0.001005  -0.029330   
1     -2.573542  -3.077104  -1.855017   2.405092  -3.763567  -2.006981   
2     -0.407573   2.449514  -1.859132  -4.070366  -6.974439   2.027302   
3      1.498563  -0.004092  -0.014200   0.000427   0.003398   0.006273   
4      1.472679  -0.012297  -0.011575   0.006366  -0.005037  -0.013553   
...         ...        ...        ...        ...        ...        ...   
8495   1.504183   0.015915   0.008294  -0.010167   0.011209   0.000540   
8496   1.491660  -0.004029  -0.028794  -0.003326  -0.005648   0.004682   
8497   1.494893  -0.006226  -0.015111   0.024876  -0.012512  -0.010283   
8498  -1.498779   0.012070   0.015826   0.000848  -0.003151   0.000662   
8499   0.434434  -4.207816   1.157510   1.283868  -2.104350   1.600885   

      feature_7  feature_8  feature_9  feature_10  label    id cluster_id  
0     -0.025058  -0.007742  -0.0048

In [14]:
print(df["cluster_id"].value_counts())

cluster_id
1_dense     2125
1_sparse    2125
0_sparse    2125
0_dense     2125
Name: count, dtype: int64


In [15]:
n0 = test_size //2
n1_big = (test_size - n0) // 2
n1_small = test_size - n0 - n1_big
n0,n1_big,n1_small

(250, 125, 125)

In [16]:
g = df.groupby("cluster_id", group_keys=False)
test_df = pd.concat([
    g.get_group("0_dense").sample(n=n1_big, random_state=seed, replace=False),
    g.get_group("0_sparse").sample(n=n1_small, random_state=seed, replace=False),
    g.get_group("1_dense").sample(n=n1_big, random_state=seed, replace=False),
    g.get_group("1_sparse").sample(n=n1_small, random_state=seed, replace=False)
]).sample(frac=1, random_state=seed)

train_df = df.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

**The most important thing in this experiment is the following code**:  
Based on the train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000] defined above, we could map the number of training samples with the following code. By choosing the number in [], we could modify the train set size. Therefore, only changing the following code block is enough to produce the experiment result successfully.

In [17]:
train_df["clean_label"] = train_df["label"].copy()

# Randomly select 20% of the training samples
noise_ratio = 0.20
noise_seed = 42

rng = np.random.default_rng(noise_seed)

n_noisy = int(len(train_df) * noise_ratio)

noisy_positions = rng.choice(
    len(train_df),
    size=n_noisy,
    replace=False
)

# Indicator showing which samples were corrupted
train_df["is_noisy"] = 0
train_df.loc[train_df.index[noisy_positions], "is_noisy"] = 1

# Flip the binary labels: 0 -> 1 and 1 -> 0
train_df.loc[
    train_df.index[noisy_positions],
    "label"
] = 1 - train_df.loc[
    train_df.index[noisy_positions],
    "label"
]

# Save a clearer name for the labels used during training
train_df["noisy_label"] = train_df["label"]

print("Number of training samples:", len(train_df))
print("Number of flipped labels:", train_df["is_noisy"].sum())
print("Noise ratio:", train_df["is_noisy"].mean())

print(
    train_df[
        ["id", "clean_label", "noisy_label", "is_noisy"]
    ].head()
)

Number of training samples: 8000
Number of flipped labels: 1600
Noise ratio: 0.2
   id  clean_label  noisy_label  is_noisy
0   1            1            1         0
1   2            1            1         0
2   3            1            1         0
3   4            1            1         0
4   5            1            1         0


In [18]:
# Feature columns used by the model
selected_features = [
    col for col in train_df.columns
    if col.startswith("feature_")
]

# Preserve IDs separately
train_ids_original = train_df["id"].to_numpy()

# Scale IDs only for the influence pipeline
IDs = (
    train_ids_original
    .reshape(-1, 1)
    .astype(np.float32)
    / 1e10
)

# Model features
X_train_features = train_df[
    selected_features
].to_numpy(dtype=np.float32)

# Append the ID column, as required by your existing influence code
X_train = np.hstack([
    X_train_features,
    IDs
])

# Use the corrupted labels for training
y_train_1d = train_df[
    "noisy_label"
].to_numpy(dtype=np.int64)

y_train = to_categorical(
    y_train_1d,
    num_classes=2
)

print(X_train.shape)
print(y_train.shape)

(8000, 11)
(8000, 2)


In [19]:
# X_train = train_df.drop(columns=["label"])
# y_train = train_df["label"]
# IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
# IDs = IDs  / 1e10

# X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
# X_train = np.hstack((X_train, IDs))
# y_train = to_categorical(y_train.values,num_classes=2)

# print(X_train)

In [20]:
X_test = test_df.drop(columns=["label","cluster_id"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)
print(y_test.shape)

(500, 11)
(500, 2)


5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [21]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [22]:
# from sklearn.metrics import pairwise_distances
# from sklearn.manifold import MDS
# import seaborn as sns
# import matplotlib.pyplot as plt

In [23]:
# D = pairwise_distances(X_all) 

In [24]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [25]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [26]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [27]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [28]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
initial_model = tf.keras.models.clone_model(model)
initial_model.set_weights(model.get_weights())
model_list.append(InfluenceModel(initial_model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  checkpoint_model = tf.keras.models.clone_model(model)
  checkpoint_model.set_weights(model.get_weights())
  model_list.append(InfluenceModel(checkpoint_model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

32/32 - 1s - loss: 0.8108 - accuracy: 0.4843 - val_loss: 0.7753 - val_accuracy: 0.2740 - 631ms/epoch - 20ms/step
32/32 - 0s - loss: 0.7549 - accuracy: 0.3717 - val_loss: 0.7218 - val_accuracy: 0.2900 - 62ms/epoch - 2ms/step
32/32 - 0s - loss: 0.7238 - accuracy: 0.4826 - val_loss: 0.6948 - val_accuracy: 0.5420 - 78ms/epoch - 2ms/step
32/32 - 0s - loss: 0.7105 - accuracy: 0.5345 - val_loss: 0.6863 - val_accuracy: 0.5440 - 58ms/epoch - 2ms/step
32/32 - 0s - loss: 0.7025 - accuracy: 0.5804 - val_loss: 0.6754 - val_accuracy: 0.8040 - 71ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6924 - accuracy: 0.6693 - val_loss: 0.6576 - val_accuracy: 0.8060 - 69ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6819 - accuracy: 0.6739 - val_loss: 0.6416 - val_accuracy: 0.8120 - 66ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6734 - accuracy: 0.6781 - val_loss: 0.6278 - val_accuracy: 0.8140 - 66ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6666 - accuracy: 0.6806 - val_loss: 0.6165 - val_accuracy: 0.8160 - 68ms/epoch - 2ms/step

In [29]:
train_logits = model.predict(
    X_train,
    batch_size=256,
    verbose=0
)

# Calculate one loss value per sample using the corrupted labels
per_sample_loss_fn = CategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)

training_losses = per_sample_loss_fn(
    y_train,
    train_logits
).numpy()

print(training_losses.shape)
print(pd.Series(training_losses).describe())

(8000,)
count    8000.000000
mean        0.576607
std         0.446798
min         0.052788
25%         0.255532
50%         0.353256
75%         0.778155
max         2.252013
dtype: float64


In [30]:
noise_loss_df = pd.DataFrame({
    "Train_ID": train_ids_original,
    "Clean_Label": train_df["clean_label"].to_numpy(),
    "Noisy_Label": train_df["noisy_label"].to_numpy(),
    "is_noisy": train_df["is_noisy"].to_numpy(),
    "Training_Loss": training_losses
})

print(noise_loss_df.head())
print(
    noise_loss_df.groupby("is_noisy")[
        "Training_Loss"
    ].describe()
)

   Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss
0         1            1            1         0       0.255569
1         2            1            1         0       0.688969
2         3            1            1         0       0.971147
3         4            1            1         0       0.255532
4         5            1            1         0       0.256884
           count      mean       std       min       25%       50%       75%  \
is_noisy                                                                       
0         6400.0  0.422145  0.287335  0.052788  0.231958  0.255998  0.530723   
1         1600.0  1.194454  0.436776  0.118174  0.806587  1.487524  1.572136   

               max  
is_noisy            
0         2.228869  
1         2.252013  


In [31]:
noise_loss_df.to_csv(
    "Noise_GroundTruth_and_TrainingLoss.csv",
    index=False
)

In [32]:
train_df.to_csv(
    "NoisyLabel_TrainingData.csv",
    index=False
)

test_df.to_csv(
    "Clean_TestData.csv",
    index=False
)

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [33]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [34]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0            1  0.163209
1            2  0.171920
2            3 -0.103400
3            4  0.163094
4            5  0.167286
...        ...       ...
7995      8496  0.165468
7996      8497 -0.560176
7997      8498  0.163094
7998      8499  0.109717
7999      8500  0.147292

[8000 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [35]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID     Score
0            1  0.006703
1            2 -0.030974
2            3 -0.003956
3            4  0.006685
4            5  0.006703
...        ...       ...
7995      8496  0.006678
7996      8497 -0.021928
7997      8498  0.006677
7998      8499  0.018781
7999      8500 -0.021889

[8000 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [36]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [37]:
TracIn_sorted.to_csv("NoisyLabel_TracIn_Scores.csv",index = False)
df_sorted.to_csv("NoisyLabel_FOIF_Scores.csv",index = False)